# Scoring — filtering, scoring, and full variant recovery

From raw barcode counts to a per-variant score table, one cell at a time, where a
**cell** is one (library, assay, treatment).

## Variant identity is a protein consequence

Every variant is named by the protein it produces, not by the barcode map's raw
`aaChanges` string. This is what lets the whole library be scored rather than only
the variants that happen to be writable as a single substitution.

It matters most for 3-nt deletions: two thirds of those in the pacybara maps start
**mid-codon**, fusing the tail of codon *n* with the head of codon *n+1*, so the map
names them with two tokens (`A350D|D351-`). Many are sequence-identical to an
ordinary single-residue deletion. Frameshifts and multi-mutants are likewise kept
and scored.

| `aaChanges` | condition | canonical identity | why |
|---|---|---|---|
| `X`*n*`Y` \| `Z`*n+1*`-` | `Y == Z` | `X`*n*`-` | X removed, Y survives |
| `X`*n*`Y` \| `Z`*n+1*`-` | `Y == X` | `Z`*n+1*`-` | Y removed, X survives |
| `X`*n*`Y` \| `Z`*n+1*`-` | `Y == *` | `X`*n*`*` | translation stops at *n* either way |
| `X`*n*`Y` \| `Z`*n+1*`-` | otherwise | `X`*n*`_Z`*n+1*`delins`*Y* | 2 residues out, 1 new in |
| `A350-` \| `D351-` | adjacent | `A350_D351del` | two-residue deletion |
| anything else with `\|` | — | kept verbatim | genuine multi-mutant |
| `K384fs` | — | kept verbatim | frameshift |

**Canonicalise first, threshold second.** Barcodes are pooled across every genotype
that produces the same protein *before* the cutoffs are applied. The other order
loses variants that only clear the mean-barcode cutoff once their degenerate routes
are summed.

**Synonymous variants are the exception.** They stay independent per-position
variants (`L155L`) and are never folded into `WT`, because they are the empirical
null the classification thresholds are built from.

## Steps

1. **Identity** — parse each `aaChanges` string, render the canonical protein-level
   name, and attach the HGVS name and position fields.
2. **The grid** — assemble the (library, assay, treatment) cells from the count
   files themselves.
3. **Filter and score**, per cell: drop barcodes shared by two variants; null any
   channel below the count cutoff; null per-replicate outliers; take the channel
   ratio; divide by wild type to get a WT-relative score; average over barcodes;
   apply the **replicate gain correction**; fit the standard curve; require at least
   5 quantifiable barcodes.
4. **Dominant-negative thresholds** — the 2.5th percentile of a bootstrap over
   empty-vector barcode means, per library and treatment. Empty-vector barcodes
   carry no kinase cassette, so they give the true "no kinase" pathway baseline
   inside each library.
5. **Drop measurements that cannot mean anything** — nonsense and frameshift in the
   six C-terminally MCP-tagged libraries for abundance and interaction, where
   truncation removes the tag before it can be translated.
6. **Write** the score table with canonical identity and reference accessions.

Position offsets are zero for this data source: the counts are built straight from
the pacybara maps' `aaChanges`, and those are already in protein numbering.

## Not covered here

- **KSR2 N-term abundance** and **SHOC2** were never assayed, so they are absent
  rather than imputed. SHOC2 has barcode maps but no experiment.
- Only the paired (PEAR-assembled) FASTQs are used. Where a run exists unpaired
  only, it is not scored.


## Setup

In [1]:
import re
import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
from scipy.optimize import curve_fit
from scipy.stats import norm

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
warnings.filterwarnings("ignore", category=RuntimeWarning)

ROOT = Path.cwd()
sys.path.insert(0, str(ROOT / "scripts"))
# The replicate gain correction lives in scripts/gain_correction.py so that this
# notebook and the QC figures share one implementation. Its docstring carries the
# derivation, the thresholds and the evidence for dropping the shape step.
from gain_correction import MAX_GAIN_DEV, correct_cell  # noqa: E402

COUNTS = ROOT / "output" / "counts" / "beforefreqcutoff"
EV_CUTOFFS = ROOT / "data" / "dn_cutoffs_empty_vector.tsv"
PROTEINS_YAML = ROOT / "config" / "proteins.yaml"
OUT = ROOT / "output" / "scoring"
OUT.mkdir(parents=True, exist_ok=True)

# config/scoring.yaml
COUNT_CUTOFF = 10
FREQ_CUTOFFS = [0, 0, 0]
MEAN_BARCODES_CUTOFF = 5
OUTLIER_SD = 2.5
OUTLIER_MIN_REPS = 2

CHANNELS = {"activity": ("pEM1", "E40"),
            "abundance": ("Flag", "HT"),
            "interaction": ("Strep", "Flag")}

STD_ACTIVITY = {"R509Y_std": 0.396, "Wild Type_std": 1.0, "G469T_std": 8.355,
                "G258E_std": 12.0, "K601D_std": 18.95}
STD_ABUNDANCE = {"E695*_std": 0.16, "I592S_std": 0.29, "G258N_std": 0.53,
                 "P367N_std": 0.71, "S727K_std": 0.66, "Wild Type_std": 1.0}
STD_EXCLUDE = {"NoVar_std", "empty_vector_std"}

# The C-terminally TAGGED libraries: SOS1, SOS2 and the RTKs EGFR, ERBB2, MET
# (HGFR) and RET. A nonsense or frameshift variant truncates the protein before
# the C-terminal MCP tag, so the tag is never translated and the abundance and
# interaction readouts measure nothing. Those rows are unscorable and Step 6
# removes them.
#
# The `_cterm`/`_nterm` suffix on the OTHER libraries does not mean this, and
# must not be added to this set. Those are alternative constructs of the same
# protein, each spanning its full length (araf_cterm 1-606, araf_nterm 1-603),
# nonsense in them is genuinely depleted, no differently from their nterm
# counterparts, so they are scorable.
#
# The exclusion covers abundance and interaction only. Activity is pEM1/E40 --
# phospho-ERK over total ERK, read out with antibodies against ERK rather than
# against the tag -- so there is no MCP tag on the protein of interest in that
# assay and truncation does not invalidate the measurement.
CTERM_MCP_LIBS = {"met", "ret", "egfr", "erbb2", "sos1", "sos2"}

PROTEIN_CFG = yaml.safe_load(PROTEINS_YAML.read_text())


def position_offset(library, assay):
    """Offset from construct numbering to protein numbering.

    Zero for this data source. `build_beforefreqcutoff.py` takes each variant
    name straight from the pacybara map's `aaChanges`, and the maps are already
    already in protein numbering -- verified by intersecting map
    identities with the scored variants directly: shp2 10,740/10,740,
    egfr 9,103/9,104, met 8,210/8,210, sos2 9,402/9,402 as-is, against 683, 0,
    0, 0 once `config/proteins.yaml: position_offsets` is added.

    The offset is real, but it belongs to the older count files, which were written
    in construct numbering. Applying it here would shift every position twice.
    """
    if COUNTS.name == "beforefreqcutoff":
        return 0
    cfg = PROTEIN_CFG.get(library, {})
    return int((cfg.get("position_offsets") or {}).get(assay, 0))


print(f"counts   : {COUNTS.relative_to(ROOT)}")
print(f"files    : {len(list(COUNTS.glob('*_dataframe_beforefreqcutoff.tsv')))}")
print(f"libraries in config: {len(PROTEIN_CFG)}")

counts   : output/counts/beforefreqcutoff
files    : 66
libraries in config: 23


## Step 1 — canonical protein-level identity

`parse_variant` turns an `aaChanges` string into a *spec*; `render` turns a spec
into an identity string with the position offset applied. The two are separate
for count files written in construct numbering rather than protein numbering, and the offset has to reach **every** position in an
identity — including both positions of a `delins` or a two-residue deletion.

The self-tests below are the specification: each is a case that the old
`process_aa_variant` either mislabelled or discarded.

In [2]:
TOKEN = re.compile(r"^([A-Z*])(\d+)([A-Z*]|-)$")
FS_TOKEN = re.compile(r"^([A-Z*])(\d+)fs$")


def _tok(t):
    m = TOKEN.match(t)
    return (m.group(1), int(m.group(2)), m.group(3)) if m else None


def parse_variant(variant):
    """aaChanges string -> (mutation type, spec)."""
    v = "" if variant is None else str(variant).strip()
    # Only a literal WT marker is wild type. An empty or unparseable aaChanges is
    # NOT -- the wild-type set is the WT normaliser's denominator, so folding an
    # unknown genotype into it shifts every score in the cell by a constant. SHP2
    # abundance has 3 such barcodes, enough to move the rep-1/rep-2 normaliser in
    # the 5th decimal and so perturb every variant in that cell.
    if v == "WT" or "_wt" in v:
        return "wild type", ("raw", "WT")
    if v in ("", "nan", "NA"):
        # A failed variant call, not a variant. pacybara emits these with
        # aaChanges/hgvsp = "NA" and hgvsc = "the condition has length > 1" (an R
        # error leaking into its output), so the genotype is unknown. Given its
        # own class so it can be dropped: a scored row whose genotype we cannot
        # state is not usable.
        return "unknown", ("raw", "unknown_genotype")
    if "std" in v:
        return "standard", ("std", v)

    parts = v.split("|")

    if len(parts) == 1:
        t = _tok(v)
        if t:
            wt, pos, mut = t
            if mut == "-":
                return "deletion", ("del", wt, pos)
            if mut == "*":
                return "nonsense", ("nonsense", wt, pos)
            if mut == wt:
                return "synonymous wild type", ("syn", wt, pos)
            return "missense", ("sub", wt, pos, mut)
        m = FS_TOKEN.match(v)
        if m:
            return "frame shift", ("fs", m.group(1), int(m.group(2)))
        return "unrecognised", ("raw", v)

    toks = [_tok(p) for p in parts]

    # a mid-codon 3-nt deletion: substitution at n plus a deletion at n+1
    if len(parts) == 2 and all(toks):
        a, b = toks
        for sub, dl in ((a, b), (b, a)):
            if dl[2] == "-" and sub[2] != "-" and dl[1] == sub[1] + 1:
                X, n, Y = sub
                Z, m = dl[0], dl[1]
                if Y == "*":
                    return "nonsense", ("nonsense", X, n)
                if Y == Z:
                    return "deletion", ("del", X, n)
                if Y == X:
                    return "deletion", ("del", Z, m)
                return "delins_2for1", ("delins2", X, n, Z, m, Y)
        if a[2] == "-" and b[2] == "-" and abs(a[1] - b[1]) == 1:
            f, s = sorted((a, b), key=lambda t: t[1])
            return "deletion_multi", ("del2", f[0], f[1], s[0], s[1])

    if all(toks):
        return "multi_change", ("multi", tuple(toks))
    if "fs" in v:
        return "multi_change_fs", ("raw", v)
    return "multi_change", ("raw", v)


def render(spec, offset=0):
    """Spec -> canonical identity in protein numbering."""
    kind = spec[0]
    if kind in ("raw", "std"):
        return spec[1]
    if kind == "sub":
        _, wt, pos, mut = spec
        return f"{wt}{pos + offset}{mut}"
    if kind == "syn":
        _, wt, pos = spec
        return f"{wt}{pos + offset}{wt}"
    if kind == "del":
        _, wt, pos = spec
        return f"{wt}{pos + offset}-"
    if kind == "nonsense":
        _, wt, pos = spec
        return f"{wt}{pos + offset}*"
    if kind == "fs":
        _, wt, pos = spec
        return f"{wt}{pos + offset}fs"
    if kind == "delins2":
        _, X, n, Z, m, Y = spec
        return f"{X}{n + offset}_{Z}{m + offset}delins{Y}"
    if kind == "del2":
        _, a, n, b, m = spec
        return f"{a}{n + offset}_{b}{m + offset}del"
    if kind == "multi":
        return "|".join(f"{wt}{pos + offset}{mut}" for wt, pos, mut in spec[1])
    raise ValueError(f"unknown spec kind {kind!r}")


def spec_fields(spec, offset=0):
    """Spec -> (Wild Type Residue, Mutation, Position)."""
    kind = spec[0]
    if kind == "std":
        return "standard", "standard", "standard"
    if kind == "raw":
        v = spec[1]
        if v == "WT":
            return "wild type", "wild type", "wild type"
        # an identity with no parseable leading token (a long insertion, or a
        # multi-change with a frameshift in it): the class is still recorded on
        # Mutation Type, but there is no single residue or position to report
        m = re.match(r"^([A-Z*])(\d+)", v)
        if m:
            return m.group(1), "complex", int(m.group(2)) + offset
        return "unknown", "unknown", "unknown"
    if kind == "sub":
        return spec[1], spec[3], spec[2] + offset
    if kind == "syn":
        return spec[1], spec[1], spec[2] + offset
    if kind == "del":
        return spec[1], "-", spec[2] + offset
    if kind == "nonsense":
        return spec[1], "*", spec[2] + offset
    if kind == "fs":
        return spec[1], "fs", spec[2] + offset
    if kind == "delins2":
        return spec[1], f"delins{spec[5]}", spec[2] + offset
    if kind == "del2":
        return spec[1], "-", spec[2] + offset
    if kind == "multi":
        first = sorted(spec[1], key=lambda t: t[1])[0]
        return first[0], "multi", first[1] + offset
    raise ValueError(kind)


AA3 = {"A": "Ala", "R": "Arg", "N": "Asn", "D": "Asp", "C": "Cys", "Q": "Gln",
       "E": "Glu", "G": "Gly", "H": "His", "I": "Ile", "L": "Leu", "K": "Lys",
       "M": "Met", "F": "Phe", "P": "Pro", "S": "Ser", "T": "Thr", "W": "Trp",
       "Y": "Tyr", "V": "Val", "*": "Ter"}
INSERTION = re.compile(r"^([A-Z*])(\d+)([A-Z]+)$")


def _hgvs_token(wt, pos, mut):
    """One change, without the `p.` prefix, in 3-letter HGVS."""
    if mut == "-":
        return f"{AA3[wt]}{pos}del"
    if mut == "*":
        return f"{AA3[wt]}{pos}Ter"
    if mut == wt:
        return f"{AA3[wt]}{pos}="
    return f"{AA3[wt]}{pos}{AA3[mut]}"


def _hgvs_part(part, offset=0):
    """One `|`-separated piece of an unparsed identity: a token or an insertion."""
    t = _tok(part)
    if t:
        return _hgvs_token(t[0], t[1] + offset, t[2])
    m = FS_TOKEN.match(part)
    if m:
        return f"{AA3[m.group(1)]}{int(m.group(2)) + offset}fs"
    m = INSERTION.match(part)     # e.g. S389SLYHH... -- an insertion
    if m:
        wt, pos, ins = m.group(1), int(m.group(2)) + offset, m.group(3)
        return f"{AA3[wt]}{pos}delins{''.join(AA3[c] for c in ins)}"
    return None


def hgvs_p(spec, offset=0):
    """Spec -> HGVS protein-level description, or None where none is meaningful.

    Rendered from our canonical spec rather than copied from the map's `hgvsp`
    column, because a pooled variant must get ONE name. `A350D|D351-` and a
    codon-aligned `A350-` are the same protein and pool into one identity here,
    but pacybara names the first `p.Ala350_Asp351delinsAsp` and the second
    `p.Ala350del` -- copying it would give one variant two names depending on
    which barcode you happened to read.

    Controls (`*_std`) and failed calls get None: neither is a sequence change.

    One known cosmetic divergence from pacybara, in `multi_change` only: it
    collapses an adjacent substitution+deletion into a delins
    (`p.[Ala591_Asp592delinsAsp;Ala602Thr]`) where this renders the tokens
    separately (`p.[Ala591Asp;Asp592del;Ala602Thr]`). Both describe the same
    protein; only the single-change classes are expected to match it exactly.
    """
    kind = spec[0]
    if kind == "std":
        return None
    if kind == "raw":
        v = spec[1]
        if v == "WT":
            return "p.="
        if v == "unknown_genotype":
            return None
        parts = [_hgvs_part(q, offset) for q in v.split("|")]
        if all(parts):
            return f"p.[{';'.join(parts)}]" if len(parts) > 1 else f"p.{parts[0]}"
        return None
    if kind == "sub":
        _, wt, pos, mut = spec
        return f"p.{_hgvs_token(wt, pos + offset, mut)}"
    if kind == "syn":
        _, wt, pos = spec
        return f"p.{AA3[wt]}{pos + offset}="
    if kind == "del":
        _, wt, pos = spec
        return f"p.{AA3[wt]}{pos + offset}del"
    if kind == "nonsense":
        _, wt, pos = spec
        return f"p.{AA3[wt]}{pos + offset}Ter"
    if kind == "fs":
        _, wt, pos = spec
        # the downstream residue and extension length are not recoverable from
        # `aaChanges`, so this is the short form -- the same one pacybara emits
        return f"p.{AA3[wt]}{pos + offset}fs"
    if kind == "delins2":
        _, X, n, Z, m_, Y = spec
        return (f"p.{AA3[X]}{n + offset}_{AA3[Z]}{m_ + offset}"
                f"delins{AA3[Y]}")
    if kind == "del2":
        _, a, n, b, m_ = spec
        return f"p.{AA3[a]}{n + offset}_{AA3[b]}{m_ + offset}del"
    if kind == "multi":
        inner = ";".join(_hgvs_token(wt, pos + offset, mut)
                         for wt, pos, mut in spec[1])
        return f"p.[{inner}]"
    raise ValueError(kind)


def canonical(variant, offset=0):
    t, spec = parse_variant(variant)
    return render(spec, offset), t


_CASES = [
    # unchanged behaviour
    ("M149F", 0, "M149F", "missense"),
    ("S25S", 0, "S25S", "synonymous wild type"),
    ("A52*", 0, "A52*", "nonsense"),
    ("A350-", 0, "A350-", "deletion"),
    ("WT", 0, "WT", "wild type"),
    ("", 0, "unknown_genotype", "unknown"),   # failed call, dropped later
    ("NA", 0, "unknown_genotype", "unknown"),
    ("Wild Type_std", 0, "Wild Type_std", "standard"),
    # previously dropped as 'unknown'
    ("A350D|D351-", 0, "A350-", "deletion"),          # A,D -> D  == delete A350
    ("S25S|A26-", 0, "A26-", "deletion"),             # S,A -> S  == delete A26
    ("Y264S|P265-", 0, "Y264_P265delinsS", "delins_2for1"),
    ("Y37*|K38-", 0, "Y37*", "nonsense"),             # truncates at 37
    ("A350-|D351-", 0, "A350_D351del", "deletion_multi"),
    ("G253A|E275D", 0, "G253A|E275D", "multi_change"),
    ("K384fs", 0, "K384fs", "frame shift"),
    # the offset reaches every position; controls are untouched
    ("A104C", 1, "A105C", "missense"),
    ("A350D|D351-", 1, "A351-", "deletion"),
    ("Y264S|P265-", 1, "Y265_P266delinsS", "delins_2for1"),
    ("G253A|E275D", 1, "G254A|E276D", "multi_change"),
    ("Wild Type_std", 1, "Wild Type_std", "standard"),
    ("WT", 1, "WT", "wild type"),
]
for _v, _off, _id, _t in _CASES:
    _got = canonical(_v, _off)
    assert _got == (_id, _t), f"{_v!r} off={_off} -> {_got}, expected {(_id, _t)}"
print(f"identity: {len(_CASES)} self-tests pass")

_HGVS_CASES = [
    ("M149F", "p.Met149Phe"),
    ("S25S", "p.Ser25="),
    ("A52*", "p.Ala52Ter"),
    ("A350-", "p.Ala350del"),
    ("WT", "p.="),
    ("K384fs", "p.Lys384fs"),
    ("A350D|D351-", "p.Ala350del"),          # pooled: one protein, one name
    ("Y264S|P265-", "p.Tyr264_Pro265delinsSer"),
    ("Y37*|K38-", "p.Tyr37Ter"),
    ("A350-|D351-", "p.Ala350_Asp351del"),
    ("G253A|E275D", "p.[Gly253Ala;Glu275Asp]"),
    ("S389SLY", "p.Ser389delinsSerLeuTyr"),  # insertion
    ("G256GGS|K268-", "p.[Gly256delinsGlyGlySer;Lys268del]"),  # insertion + del
    ("Wild Type_std", None),                 # a control, not a change
    ("", None),                              # failed call
]
for _v, _h in _HGVS_CASES:
    _got = hgvs_p(parse_variant(_v)[1])
    assert _got == _h, f"hgvs_p({_v!r}) -> {_got!r}, expected {_h!r}"
print(f"hgvs: {len(_HGVS_CASES)} self-tests pass")

identity: 21 self-tests pass
hgvs: 15 self-tests pass


## Step 2 — the count files, and the experimental grid

Filenames give library, assay and treatment. A file with no treatment suffix
means the untreated arm, recorded as `No_treatment` — with two
exceptions that would otherwise mismatch silently, so they are declared and then
**asserted** rather than trusted.

In [3]:
CONDITION_NORM = {"dmso": "DMSO", "hsp90i": "HSP90i", "serumstarve": "SerumStarve",
                  "lztr1kociar": "LZTR1koCIAR", "lztr1ko": "LZTR1ko",
                  "lztr1": "LZTR1", "ciar": "CIAR"}
_COND = "|".join(map(re.escape, ["LZTR1koCIAR", "LZTR1ko", "LZTR1", "CIAR",
                                 "HSP90i", "SerumStarve", "dmso", "DMSO",
                                 "Unknown"]))
_SUF = r"_dataframe_beforefreqcutoff\.tsv"
_PATTERNS = [
    re.compile(rf"^(?P<d>[A-Za-z0-9]{{6,8}})_(.+?)_(nterm|cterm)_(activity|abundance|interaction)_({_COND}){_SUF}$", re.I),
    re.compile(rf"^(?P<d>[A-Za-z0-9]{{6,8}})_(.+?)_(nterm|cterm)_(activity|abundance|interaction){_SUF}$", re.I),
    re.compile(rf"^(?P<d>[A-Za-z0-9]{{6,8}})_(.+?)_(activity|abundance|interaction)_({_COND}){_SUF}$", re.I),
    re.compile(rf"^(?P<d>[A-Za-z0-9]{{6,8}})_(.+?)_(activity|abundance|interaction){_SUF}$", re.I),
]

# build_beforefreqcutoff.py already names files with the canonical treatment
# labels, so nothing needs renaming here. The map is kept (empty) so the
# assertion below still reports any cell that fails to line up.
TREATMENT_OVERRIDE: dict[tuple, str] = {}

def parse_count_filename(name):
    for i, pat in enumerate(_PATTERNS):
        m = pat.match(name)
        if not m:
            continue
        g = m.groups()
        if i == 0:
            date, protein, term, assay, cond = g
            lib = f"{protein.lower()}_{term.lower()}"
        elif i == 1:
            date, protein, term, assay = g
            lib, cond = f"{protein.lower()}_{term.lower()}", None
        elif i == 2:
            date, protein, assay, cond = g
            lib = protein.lower()
        else:
            date, protein, assay = g
            lib, cond = protein.lower(), None
        assay = assay.lower()
        treat = CONDITION_NORM.get(cond.lower(), cond) if cond else "No_treatment"
        treat = TREATMENT_OVERRIDE.get((lib, assay, treat), treat)
        return date, lib, assay, treat
    return None


files = sorted(COUNTS.glob("*_dataframe_beforefreqcutoff.tsv"))
inventory = []
for f in files:
    parsed = parse_count_filename(f.name)
    if parsed is None:
        print(f"UNPARSEABLE: {f.name}")
        continue
    date, lib, assay, treat = parsed
    inventory.append({"file": f.name, "run_date": date, "library": lib,
                      "assay": assay, "assay_treatment": treat,
                      "offset": position_offset(lib, assay), "path": f})
inventory = pd.DataFrame(inventory)
print(f"{len(inventory)} count files parsed")

# A cell can have been sequenced more than once. Grouping is by
# (library, assay, assay_treatment) and *not* by date, so both runs fall into
# one group and their barcode rows are concatenated -- a barcode seen in two
# runs contributes one row per run, each with its own ratio, and the per-variant
# mean averages across both. They are extra replicates of one condition, not
# duplicate copies of it.
#
# This matters: MET is the only twice-sequenced library, and pooling its runs
# rather than picking one is what makes its two arms reproducible.
CELL = ["library", "assay", "assay_treatment"]
dup_mask = inventory.duplicated(CELL, keep=False)
if dup_mask.any():
    print("\ncells sequenced more than once -- their runs are pooled:")
    print(inventory.loc[dup_mask, ["run_date", "file"] + CELL]
          .sort_values(CELL + ["run_date"]).to_string(index=False))
print(f"\n{len(inventory)} count files -> {inventory.groupby(CELL).ngroups} cells\n")

# The experimental grid, from the count files alone -- the data decide which
# cells exist, and the assertion below catches a cell that fails to line up.
ours = sorted(set(map(tuple, inventory[["library", "assay",
                                        "assay_treatment"]].values)))
print(f"cells built from counts: {len(ours)}")
_by_assay = {}
for lib, assay, treat in ours:
    _by_assay.setdefault(assay, []).append(f"{lib}/{treat}")
for assay in sorted(_by_assay):
    print(f"  {assay:12s} {len(_by_assay[assay]):>2d} cells")

# Treatments must all be canonical, or a typo silently becomes a new cell.
_bad = sorted({t for _, _, t in ours} - set(CONDITION_NORM.values())
              - {"No_treatment"})
assert not _bad, f"non-canonical treatment label(s): {_bad}"

print("\noffsets in use:")
print(inventory.groupby(["library", "assay"])["offset"].first()
      .loc[lambda s: s != 0].to_string())

66 count files parsed

cells sequenced more than once -- their runs are pooled:
run_date                                                       file library     assay assay_treatment
  251010   251010_met_abundance_DMSO_dataframe_beforefreqcutoff.tsv     met abundance            DMSO
  251211   251211_met_abundance_DMSO_dataframe_beforefreqcutoff.tsv     met abundance            DMSO
  251010 251010_met_abundance_HSP90i_dataframe_beforefreqcutoff.tsv     met abundance          HSP90i
  251211 251211_met_abundance_HSP90i_dataframe_beforefreqcutoff.tsv     met abundance          HSP90i

66 count files -> 64 cells

cells built from counts: 64
  abundance    39 cells
  activity     24 cells
  interaction   1 cells

offsets in use:
Series([], )


## Step 3 — filtering and scoring

Per cell, in this order:

1. drop barcodes mapping to more than one variant in the group
2. counts below `count_cutoff` → NaN, per channel per replicate
3. frequency filter (cutoffs are `[0, 0, 0]`, so a no-op — kept for fidelity)
4. `ratio_j = numerator_j / denominator_j`
5. outlier barcodes — ratio above the variant's median + 2.5 SD in ≥2 replicates
   → NaN in **all** replicates. Upper tail only
6. `score_j` = ratio normalised to the mean WT ratio
7. aggregate to variants, then variant frequency from total reads
8. standard curve through the origin, fitted per replicate on the BRAF spikes
9. `average_num_quant_bc`, then the `>= 5` cutoff (standards exempt)
10. classification from the 2.5/97.5 percentiles of the synonymous distribution

In [4]:
def filter_and_score(data, assay, library, treatment):
    """One (library, assay, treatment) cell: barcodes in, variant scores out."""
    numerator, denominator = CHANNELS[assay]

    nuniq = data.groupby("barcode")["variant"].nunique()
    dupes = nuniq[nuniq > 1].index
    if len(dupes):
        data = data[~data["barcode"].isin(dupes)].copy()

    data["Number of Barcodes"] = data["variant"].map(data["variant"].value_counts())

    for j in (1, 2, 3):
        for ch in (denominator, numerator):
            data[f"{ch}_{j}"] = np.where(data[f"{ch}_{j}"] < COUNT_CUTOFF,
                                         np.nan, data[f"{ch}_{j}"])

    for value, j in zip(FREQ_CUTOFFS, (1, 2, 3)):
        tot = data[f"{denominator}_{j}"].sum() + data[f"{numerator}_{j}"].sum()
        freq = (data[f"{denominator}_{j}"] + data[f"{numerator}_{j}"]) / tot
        for ch in (denominator, numerator):
            data[f"{ch}_{j}"] = np.where(freq < value, np.nan, data[f"{ch}_{j}"])

    for j in (1, 2, 3):
        data[f"ratio_{j}"] = data[f"{numerator}_{j}"] / data[f"{denominator}_{j}"]

    flags = []
    for j in (1, 2, 3):
        med = data.groupby("variant")[f"ratio_{j}"].transform("median")
        sd = data.groupby("variant")[f"ratio_{j}"].transform("std")
        flags.append(data[f"ratio_{j}"] > med + OUTLIER_SD * sd)
    outliers = sum(f.astype(int) for f in flags) >= OUTLIER_MIN_REPS
    for j in (1, 2, 3):
        data.loc[outliers, [f"{denominator}_{j}", f"{numerator}_{j}",
                            f"ratio_{j}"]] = np.nan

    data["average ratio"] = data[[f"ratio_{j}" for j in (1, 2, 3)]].mean(axis=1)

    wt = data[data["Mutation Type"] == "wild type"]
    for j in (1, 2, 3):
        wt_ratio = (wt[f"{numerator}_{j}"] / wt[f"{denominator}_{j}"]).mean()
        data[f"score_{j}"] = (data[f"{numerator}_{j}"]
                              / data[f"{denominator}_{j}"]) / wt_ratio

    agg = ["Number of Barcodes", "ratio_1", "ratio_2", "ratio_3",
           "average ratio", "score_1", "score_2", "score_3"]
    scores = data.groupby("variant")[agg].mean().reset_index()

    for j in (1, 2, 3):
        data[f"reads_{j}"] = (data[f"{denominator}_{j}"].fillna(0)
                              + data[f"{numerator}_{j}"].fillna(0))
    data["total_reads"] = data[[f"reads_{j}" for j in (1, 2, 3)]].sum(axis=1)
    vr = data.groupby("variant")["total_reads"].sum().reset_index()
    vr["variant_frequency"] = vr["total_reads"] / data["total_reads"].sum()
    scores = scores.merge(vr[["variant", "variant_frequency"]], on="variant")

    meta = (data.groupby("variant")[["Mutation Type", "Wild Type Residue",
                                     "Mutation", "Position", "hgvs_p"]]
            .first().reset_index())
    scores = scores.merge(meta, on="variant")

    counts = (data.groupby("variant")[[f"score_{j}" for j in (1, 2, 3)]]
              .count().reset_index())
    counts["average_num_quant_bc"] = counts[
        [f"score_{j}" for j in (1, 2, 3)]].mean(axis=1)
    scores = scores.merge(counts[["variant", "average_num_quant_bc"]], on="variant")

    # ---- replicate gain correction -------------------------------------------
    # One exponent, clamped, applied only to a replicate that resolves materially
    # less of this cell's range than its two siblings do. Across all 192
    # replicates exactly two qualify -- KSR1 C-term activity rep 3 and MRAS
    # activity rep 2 -- and every other replicate comes through bit-identical, so
    # `corrected_score_j` equals `score_j` everywhere else. See
    # scripts/gain_correction.py for the method and the thresholds.
    #
    # The gain is fitted on missense variants that clear the barcode cutoff: the
    # population it is best measured in. It is then applied to EVERY class, and to
    # the standards and the empty-vector controls below, so that one cell's
    # replicate is never left with its variants on one scale and its controls on
    # another.
    fit_on = ((scores["Mutation Type"] == "missense")
              & (scores["average_num_quant_bc"] >= MEAN_BARCODES_CUTOFF))
    corrected, corr_log, corr_maps = correct_cell(
        scores, [f"score_{j}" for j in (1, 2, 3)], fit_on,
        cell={"library": library, "assay": assay, "treatment": treatment})
    for _j in (1, 2, 3):
        scores[f"corrected_score_{_j}"] = corrected[:, _j - 1]

    # Everything downstream is built from the CORRECTED replicates: the average
    # score, the standard curve, the DN classification and the empty-vector
    # threshold. `score_j` is kept unchanged alongside as the raw record.
    scores["average score"] = scores[
        [f"corrected_score_{_j}" for _j in (1, 2, 3)]].mean(axis=1)

    assigned = (STD_ACTIVITY if assay == "activity"
                else STD_ABUNDANCE if assay == "abundance" else {})
    # Per replicate, one parameter: y = m x through the origin, fitted over the
    # assigned standards.
    #
    # Fitted TWICE. `intercept_0_std_adj_score_j` is the canonical one and comes
    # from the CORRECTED replicates, whose standards went through the same map as
    # the variants, so numerator and calibration are consistent. `raw_std_adj_
    # score_j` repeats it on the raw replicates and is kept only so the
    # correction's effect on the calibrated scale can be seen -- nothing
    # downstream reads it.
    for _tag, _src in (("intercept_0_std_adj_score", "corrected_score"),
                       ("raw_std_adj_score", "score")):
        for _j in (1, 2, 3):
            rep = f"{_src}_{_j}"
            std = scores[(scores["Mutation Type"] == "standard")
                         & (~scores["variant"].isin(STD_EXCLUDE))].copy()
            std["assigned"] = pd.to_numeric(std["variant"].map(assigned),
                                            errors="coerce")
            std = std.dropna(subset=["assigned", rep])
            if std["assigned"].notna().sum() >= 2:
                slope = curve_fit(lambda x, m: m * x,
                                  std["assigned"].values, std[rep].values)[0][0]
                scores[f"{_tag}_{_j}"] = scores[rep] / slope
            else:
                scores[f"{_tag}_{_j}"] = np.nan
    scores["intercept_0_standard-adjusted score"] = scores[
        [f"intercept_0_std_adj_score_{j}" for j in (1, 2, 3)]].mean(axis=1)

    is_std = scores["Mutation Type"] == "standard"
    scores = scores[is_std
                    | (scores["average_num_quant_bc"] >= MEAN_BARCODES_CUTOFF)].copy()

    scores["library"] = library
    scores["assay"] = assay
    scores["assay_treatment"] = treatment

    # BRAF activity is exempt from the standard curve because
    # BRAF is the source of the standards -- they were never spiked into the BRAF
    # activity libraries, so no curve can be fitted and the raw WT-normalised
    # score is substituted.
    # Without this the column is NaN for every BRAF activity row, and BRAF drops
    # silently out of any analysis keyed on the standard-adjusted score.
    if assay == "activity" and library in ("braf_cterm", "braf_nterm"):
        for _j in (1, 2, 3):
            scores[f"intercept_0_std_adj_score_{_j}"] = \
                scores[f"corrected_score_{_j}"]
            scores[f"raw_std_adj_score_{_j}"] = scores[f"score_{_j}"]
        scores["intercept_0_standard-adjusted score"] = scores["average score"]

    syn = scores[scores["Mutation Type"] == "synonymous wild type"]["average score"]
    lo, hi = (syn.quantile(0.025), syn.quantile(0.975)) if len(syn) else (np.nan, np.nan)
    scores["classification_2.5pct"] = [
        None if pd.isna(s) or pd.isna(lo) or pd.isna(hi)
        else "low" if s < lo else "high" if s > hi else "wt-like"
        for s in scores["average score"]]

    # Empty-vector barcodes carry the DN threshold and are only scored here, so
    # hand them back rather than re-reading the file in the DN step. Strictly
    # `empty_vector_std`: the canonical DN threshold is the empty-vector one, and
    # the older `NoVar_std` control it replaced must not be silently substituted.
    ev = data[data["variant"].isin(["empty_vector_std", "NoVar_std"])][
        ["variant", "score_1", "score_2", "score_3"]].copy()
    # the control barcodes ride the same map as the variants, so the DN threshold
    # and the scores it is compared against stay on one scale
    for _j in (1, 2, 3):
        _f = corr_maps[_j - 1]
        ev[f"corrected_score_{_j}"] = (ev[f"score_{_j}"] if _f is None
                                       else _f(ev[f"score_{_j}"].to_numpy()))
    # `variant` distinguishes the two controls; the DN step keeps them apart
    ev["library"] = library
    ev["assay"] = assay
    ev["assay_treatment"] = treatment
    return scores, ev, corr_log


def load_cell(row):
    """Read one count file and attach canonical identities."""
    df = pd.read_csv(row["path"], sep="\t")
    if "barcode" not in df.columns:
        # 35 of the 64 files were written without the barcode column; only its
        # uniqueness is used (to drop barcodes shared by two variants), and a
        # file with no barcodes has none to drop.
        df.insert(0, "barcode", np.arange(len(df)))
    off = row["offset"]
    uniq = df["variant"].astype(str).unique()
    lut = {}
    for v in uniq:
        t, spec = parse_variant(v)
        lut[v] = ((render(spec, off), t) + spec_fields(spec, off)
                  + (hgvs_p(spec, off),))
    key = df["variant"].astype(str)
    df["variant"] = key.map(lambda v: lut[v][0])
    df["Mutation Type"] = key.map(lambda v: lut[v][1])
    df["Wild Type Residue"] = key.map(lambda v: lut[v][2])
    df["Mutation"] = key.map(lambda v: lut[v][3])
    df["Position"] = key.map(lambda v: lut[v][4])
    # attached here, not in Step 9, because the HGVS name is rendered from the
    # *spec* and the canonical identity string cannot be parsed back into one.
    # Pooling is safe: identical canonical identity implies identical spec.
    df["hgvs_p"] = key.map(lambda v: lut[v][5])
    return df

## Step 4 — run every cell

Each file is scored independently; nothing is shared between cells except the
constants. Per-file counts of barcodes and recovered variants are printed so a
cell that behaves oddly is visible rather than averaged away.

In [5]:
results, per_file, ev_frames, corr_records = [], [], [], []
for _cell_key, cell_rows in inventory.groupby(CELL, sort=False):
    row = cell_rows.iloc[0]
    # concatenate every run of this cell. Barcodes are deliberately left as they
    # are, so one seen in two runs appears twice.
    df = pd.concat([load_cell(r) for _, r in cell_rows.iterrows()],
                   ignore_index=True)
    n_bc = len(df)
    n_compound = int(df["Mutation Type"].isin(
        ["delins_2for1", "multi_change", "multi_change_fs", "frame shift",
         "deletion_multi", "unrecognised"]).sum())
    scores, ev, corr_log = filter_and_score(df, row["assay"], row["library"],
                                            row["assay_treatment"])
    results.append(scores)
    corr_records += corr_log
    if len(ev):
        ev_frames.append(ev)
    per_file.append({
        "library": row["library"], "assay": row["assay"],
        "treatment": row["assay_treatment"], "barcodes": n_bc,
        "bc_previously_dropped": n_compound,
        "variants": len(scores),
        "recovered": int((~scores["Mutation Type"].isin(
            ["missense", "synonymous wild type", "nonsense", "deletion",
             "wild type", "standard"])).sum()),
    })
    print(f"  {row['library']:12s} {row['assay']:11s} {row['assay_treatment']:12s} "
          f"bc={n_bc:8,d}  variants={len(scores):7,d}  "
          f"recovered={per_file[-1]['recovered']:6,d}")

scored = pd.concat(results, ignore_index=True)
per_file = pd.DataFrame(per_file)
print(f"\ntotal variant effects scored: {len(scored):,}")

# ---- the replicate gain correction, as applied ------------------------------
correction_log = pd.DataFrame(corr_records)
correction_log.to_csv(OUT / "replicate_correction_log.tsv", sep="\t", index=False)
_c = correction_log[correction_log.status.str.startswith("corrected")]
print(f"\ngain correction: {len(_c)} of {len(correction_log)} replicates corrected "
      f"(threshold {MAX_GAIN_DEV} log2); the other "
      f"{len(correction_log) - len(_c)} are bit-identical to their input")
if len(_c):
    print(_c[["library", "assay", "treatment", "rep", "gain", "gain_dev_log2",
              "gain_ratio", "clamp_lo", "clamp_hi", "dev_before", "dev_after",
              "shift_log2", "reference_reps"]].round(4).to_string(index=False))
_n = (scored[[f"corrected_score_{j}" for j in (1, 2, 3)]].to_numpy()
      != scored[[f"score_{j}" for j in (1, 2, 3)]].to_numpy()).any(axis=1).sum()
print(f"  rows whose corrected score differs from the raw score: {_n:,} "
      f"of {len(scored):,}")

  kras         abundance   LZTR1koCIAR  bc= 201,351  variants=  4,238  recovered=   256


  kras         abundance   LZTR1ko      bc= 201,245  variants=  4,238  recovered=   256


  kras         activity    CIAR         bc= 347,091  variants=  4,336  recovered=   272


  kras         activity    DMSO         bc= 371,285  variants=  4,343  recovered=   277


  kras         abundance   No_treatment bc= 205,859  variants=  4,313  recovered=   271


...

  craf_nterm   abundance   HSP90i       bc= 158,703  variants=  7,318  recovered=   470


  araf_nterm   abundance   HSP90i       bc= 245,574  variants=  6,590  recovered=   877


  ret          abundance   HSP90i       bc= 494,920  variants= 10,918  recovered= 1,067


  braf_cterm   abundance   CIAR         bc= 142,102  variants=  7,734  recovered=   517


  braf_nterm   abundance   CIAR         bc= 174,842  variants=  8,967  recovered=   944

total variant effects scored: 541,131

gain correction: 2 of 192 replicates corrected (threshold 0.5 log2); the other 190 are bit-identical to their input
   library    assay    treatment  rep   gain  gain_dev_log2  gain_ratio  clamp_lo  clamp_hi  dev_before  dev_after  shift_log2 reference_reps
      mras activity No_treatment    2 0.5703         0.8103      1.9790    0.5675    1.2809      0.6146     0.1799      0.1628            1+3
ksr1_cterm activity No_treatment    3 0.5077         0.9779      2.0029    0.8073    3.5362      1.4728     0.1884      0.1812            1+2
  rows whose corrected score differs from the raw score: 19,568 of 541,131


## Step 5 — dominant-negative thresholds

A dominant-negative variant is one whose activity is at or below the level of a
cell carrying no kinase at all. **Empty-vector barcodes** measure that level
directly: they carry no ORF, so their activity is the pathway's baseline inside
that library, under that treatment.

The threshold is the 2.5th percentile of a bootstrap over empty-vector barcode
means — resample `BOOT_K` barcodes, take their mean, repeat `BOOT_N` times, read
the percentile off the draw distribution. It bounds a barcode *average* rather
than an individual barcode, which is the right comparison because a variant score
is itself a mean over its barcodes.

2.5% in one tail: the synonymous test used for `classification_2.5pct` is
two-sided, so 2.5% each side there is the same evidentiary bar as 2.5% in this
one-sided tail.

`data/dn_cutoffs_empty_vector.tsv` supplies every cell, and a live bootstrap
overrides the cells whose count files actually carry `empty_vector_std` barcodes.
Only `empty_vector_std` is accepted — the older `NoVar_std` control is reported
alongside for reference but never substituted, because it sits at a different
level and would silently rebuild a superseded threshold.

The call itself is made in stage 2 as `DN_EV`, which also applies the
inhibitory-protein exclusion.


In [6]:
# 5th, not 2.5th: the synonymous test is two-sided (a variant may be high or
# low), but the empty-vector test is one-sided -- we only ask whether a variant
# is at or below the no-protein level. 2.5% each side of a two-sided test is
# the same evidentiary bar as 5% in one tail.
N_BOOTSTRAP, N_BARCODES, PERCENTILE, SEED = 200, 10, 2.5, 42


def _bootstrap(frame):
    """2.5th percentile of a bootstrap over the given control's barcodes.

    Resample `N_BARCODES` empty-vector barcodes, take their mean, repeat
    `N_BOOTSTRAP` times, and read the `PERCENTILE`th percentile of those means.

    This is the original procedure, kept deliberately after an investigation of
    the alternatives. The key point is that it bounds a barcode *average*, not an
    individual barcode: a variant score is itself a mean over its barcodes (median
    15 in the activity libraries), so a bound on a single barcode -- e.g. the 2.5th
    percentile of a lognormal fitted to the per-barcode scores -- is the wrong
    comparison and lands roughly 1.5x too low. Resampling 10 barcodes puts the
    null on the same footing as the quantity being tested.

    Two external checks support the thresholds it produces. R509Y, the
    dimerisation-deficient BRAF dominant negative, falls below the threshold in
    every library where it is measurable -- as a spike-in in most, and as an
    ordinary library variant in braf_cterm, where its measured score (0.395)
    reproduces the assigned standard value (0.396). And `NoVar_std`, the other
    no-protein control, coincides with the empty-vector level (median ratio 1.05),
    which is what two independent no-protein controls should do.

    A refinement left for later: matching `N_BARCODES` to each variant's own
    barcode count rather than fixing it at 10, which would tighten the threshold
    for well-measured variants and loosen it for marginal ones.
    """
    out = []
    for (lib, treat), grp in frame.groupby(["library", "assay_treatment"]):
        rng = np.random.default_rng(SEED)
        s = grp[[f"score_{j}" for j in (1, 2, 3)]].to_numpy(dtype=float)
        n = len(grp)
        boot = np.array([
            np.nanmean(np.nanmean(
                s[rng.choice(n, N_BARCODES, replace=n < N_BARCODES)], axis=0))
            for _ in range(N_BOOTSTRAP)])
        out.append({"library": lib, "assay_treatment": treat,
                    "dn_threshold": np.percentile(boot, PERCENTILE),
                    "n_ev_barcodes": n})
    return out


ev_rows, novar_rows = [], []
if ev_frames:
    ev_all = pd.concat(ev_frames, ignore_index=True)
    ev_all = ev_all[ev_all["assay"] == "activity"]
    # the canonical threshold is the empty-vector one; NoVar_std is the older
    # control it replaced and is only computed to show what it would have given
    ev_rows = _bootstrap(ev_all[ev_all["variant"] == "empty_vector_std"])
    novar_rows = _bootstrap(ev_all[ev_all["variant"] == "NoVar_std"])
    print(f"empty_vector_std barcodes: "
          f"{int((ev_all['variant'] == 'empty_vector_std').sum()):,}; "
          f"NoVar_std: {int((ev_all['variant'] == 'NoVar_std').sum()):,}")

# Start from the canonical precomputed thresholds, then let a live bootstrap
# override any cell whose count file actually carries empty-vector barcodes.
# Per-cell rather than either/or, so a partial delivery still gets full coverage.
thresholds = pd.read_csv(EV_CUTOFFS, sep=r"\s+")
thresholds["source"] = "precomputed (data/dn_cutoffs_empty_vector.tsv)"
if ev_rows:
    boot = pd.DataFrame(ev_rows)
    boot["source"] = "bootstrap from empty_vector_std barcodes"
    thresholds = pd.concat(
        [boot, thresholds[~thresholds.set_index(["library", "assay_treatment"])
                          .index.isin(boot.set_index(
                              ["library", "assay_treatment"]).index)]],
        ignore_index=True)
print("DN threshold source:")
print(thresholds["source"].value_counts().to_string())

# The bootstrap that IS reproducible here runs on NoVar_std, the control the
# empty vector replaced. It is not used -- shown so the size of the substitution
# we are declining to make is visible rather than assumed.
if novar_rows:
    nv = pd.DataFrame(novar_rows).rename(columns={"dn_threshold": "novar_threshold"})
    cmp_ = thresholds.merge(nv[["library", "assay_treatment", "novar_threshold",
                                "n_ev_barcodes"]],
                            on=["library", "assay_treatment"], how="inner",
                            suffixes=("", "_nv"))
    if len(cmp_):
        cmp_["ratio"] = cmp_["novar_threshold"] / cmp_["dn_threshold"]
        print(f"\nNoVar_std bootstrap vs the canonical empty-vector threshold "
              f"({len(cmp_)} cells with both):")
        print(cmp_[["library", "assay_treatment", "dn_threshold",
                    "novar_threshold", "ratio"]].to_string(index=False))
        print(f"  median NoVar/EV ratio: {cmp_['ratio'].median():.2f}")

# The thresholds are reported and written, but NO classification column is
# emitted here. The DN call is `DN_EV`, made in stage 2 by
# `annotation.py: add_dominant_negative_ev`: it applies the inhibitory-protein
# exclusion, and it is NaN rather than False where a cell has no usable
# empty-vector baseline, so "not dominant-negative" and "not assessable" stay
# distinguishable.
_thr = thresholds[["library", "assay_treatment", "dn_threshold"]]
_act = scored[scored["assay"] == "activity"].merge(
    _thr, on=["library", "assay_treatment"], how="left")
print(f"\nactivity rows below the empty-vector threshold "
      f"(for reference; the call itself is DN_EV): "
      f"{int((_act['average score'] < _act['dn_threshold']).sum()):,}"
      f" of {len(_act):,}")
del _act, _thr

empty_vector_std barcodes: 69,242; NoVar_std: 9,357
DN threshold source:
source
bootstrap from empty_vector_std barcodes          24
precomputed (data/dn_cutoffs_empty_vector.tsv)     1

NoVar_std bootstrap vs the canonical empty-vector threshold (24 cells with both):
   library assay_treatment  dn_threshold  novar_threshold     ratio
araf_cterm    No_treatment      0.637192         0.618510  0.970682
araf_nterm    No_treatment      0.561058         0.497646  0.886979
braf_cterm    No_treatment      0.899719              NaN       NaN
braf_nterm    No_treatment      0.735303         1.109125  1.508392
craf_cterm    No_treatment      0.258778         0.244479  0.944744
craf_nterm    No_treatment      0.164772         0.166454  1.010208
      egfr    No_treatment      0.171557         0.168702  0.983356
      egfr     SerumStarve      0.736745        10.963612 14.881146
     erbb2    No_treatment      0.351214         0.330594  0.941289
      grb2    No_treatment      1.254744           

## Step 6 — drop the measurements that cannot mean anything

Nonsense in a C-terminal-MCP library truncates the protein before the tag, so
abundance and interaction readouts for it are artifacts. This covers every
class that introduces a stop, recovered ones included: a
`delins`-to-stop is a nonsense variant and is already canonicalised to `X`*n*`*`,
so it is caught by the same rule. Frameshifts in those libraries lose the tag
for the same reason and go with them.

In [7]:
# `library` is matched exactly, not suffix-stripped: the six C-terminally tagged
# libraries carry no `_nterm`/`_cterm` suffix, and stripping one would let an
# N-terminally tagged construct of the same protein match this set by mistake.
truncating = scored["Mutation Type"].isin(["nonsense", "frame shift",
                                           "multi_change_fs"])
unscorable = (scored["library"].isin(CTERM_MCP_LIBS) & truncating
              & scored["assay"].isin(["abundance", "interaction"]))
print(f"unscorable -- truncates before the C-terminal MCP tag, so the tag is "
      f"never translated: {int(unscorable.sum()):,} variant effects")
print(scored[unscorable].groupby(["library", "assay", "Mutation Type"])
      .size().to_string())
# Written out rather than silently discarded, so the exclusion is auditable and
# the count can be reconciled against the pre-exclusion table.
_uns = scored[unscorable].copy()
# Carries its own classification: these rows never reach Step 9, where
# `variant_category` is assigned, so the label is set here explicitly.
_uns["variant_category"] = "unscorable"
_uns["exclusion_reason"] = ("truncates before the C-terminal MCP tag, so the "
                            "abundance/interaction readout measures nothing")

# A second, disjoint class of unscorable row: a variant at a position OUTSIDE the
# reference protein. Those are construct-junction artifacts, not protein variants.
# The C-terminally tagged ORFs have their native stop removed so they can fuse to
# the linker and MCP, so a frameshift called one past the last residue -- ERBB2
# `T1256fs`, where Thr is the linker's first residue -- breaks the *tag*, leaving
# the protein itself wild type. That position carries only frameshifts, and far
# fewer barcodes than the neighbouring designed positions.
#
# The native stop at length+1 is a different thing and is KEPT: in an
# N-terminally tagged library that position is a real `*`, and a frameshift there
# (braf_nterm `*767fs`, kras `*189fs`) is a genuine readthrough of the real
# protein. Hence the `!= "*"` clause rather than a blanket cut at `length`.
ACC_LEN = {q: v["length"] for q, v in
           yaml.safe_load((ROOT / "config"
                           / "protein_accessions.yaml").read_text()).items()
           if (v or {}).get("length")}
_prot = scored["library"].str.replace(r"_(nterm|cterm)$", "", regex=True)
_pos = pd.to_numeric(scored["Position"], errors="coerce")
_ref = _prot.map(ACC_LEN)
outside = (_pos.notna() & _ref.notna()
           & ((_pos > _ref + 1)
              | ((_pos == _ref + 1) & (scored["Wild Type Residue"] != "*"))))
if outside.any():
    print(f"\nunscorable -- position lies outside the reference protein "
          f"(construct junction): {int(outside.sum()):,} variant effects")
    print(scored.loc[outside, ["library", "assay", "variant", "Position",
                               "Wild Type Residue", "Mutation Type",
                               "Number of Barcodes"]].to_string(index=False))
_out = scored[outside].copy()
_out["variant_category"] = "unscorable"
_out["exclusion_reason"] = ("position lies outside the reference protein -- a "
                            "construct-junction artifact, so the protein itself "
                            "is wild type")

pd.concat([_uns, _out], ignore_index=True).to_csv(
    OUT / "unscorable_cterm_mcp.tsv", sep="\t", index=False)
print(f"  -> {(OUT / 'unscorable_cterm_mcp.tsv').relative_to(ROOT)} "
      f"({int(unscorable.sum()) + int(outside.sum()):,} rows)")
scored = scored[~(unscorable | outside)].copy()

# Variants whose genotype we cannot state are dropped outright. A score is only
# useful attached to a known sequence change, and these are pacybara call
# failures rather than a variant class.
unknown = scored["Mutation Type"] == "unknown"
if unknown.any():
    print(f"\ndropping {int(unknown.sum()):,} variant effects with an unknown "
          f"genotype (failed pacybara variant call)")
    print(scored.loc[unknown, ["library", "assay", "assay_treatment",
                               "Number of Barcodes"]].to_string(index=False))
    scored = scored[~unknown].copy()
else:
    print("\nno unknown-genotype rows survived the barcode cutoff")

print(f"\nvariant effects after exclusion: {len(scored):,}")

unscorable -- truncates before the C-terminal MCP tag, so the tag is never translated: 9,331 variant effects
library  assay      Mutation Type
egfr     abundance  frame shift       790
                    nonsense          761
erbb2    abundance  frame shift       861
                    nonsense          354
met      abundance  frame shift      1450
                    nonsense          862
ret      abundance  frame shift      1490
                    nonsense          903
sos1     abundance  frame shift       475
                    nonsense          400
sos2     abundance  frame shift       340
                    nonsense          645



unscorable -- position lies outside the reference protein (construct junction): 1 variant effects
library    assay variant Position Wild Type Residue Mutation Type  Number of Barcodes
  erbb2 activity T1256fs     1256                 T   frame shift                11.0


  -> output/scoring/unscorable_cterm_mcp.tsv (9,332 rows)

dropping 2 variant effects with an unknown genotype (failed pacybara variant call)
library    assay assay_treatment  Number of Barcodes
   mek1 activity    No_treatment                11.0
   shp2 activity    No_treatment                18.0



variant effects after exclusion: 531,797


## Step 8 — what the recovery adds

`recovered` counts variant effects in classes the old pipeline could not name at
all. `pooled` counts variants that existed before but whose barcode support grew
because degenerate genotypes were summed into them — those did not gain a row,
they gained precision, and some crossed `mean_barcodes_cutoff` for the first
time.

In [8]:
NEW_CLASSES = ["delins_2for1", "frame shift", "multi_change", "multi_change_fs",
               "deletion_multi", "unrecognised"]
is_new = scored["Mutation Type"].isin(NEW_CLASSES)

summary = (scored.assign(recovered=is_new)
           .groupby("Mutation Type")
           .agg(variant_effects=("variant", "size"),
                distinct_variants=("variant", "nunique"),
                median_barcodes=("Number of Barcodes", "median"))
           .sort_values("variant_effects", ascending=False))
print("variant effects by mutation type\n")
print(summary.to_string())

print(f"\nrecovered (previously unnameable) variant effects: {int(is_new.sum()):,}")
print(f"total variant effects                            : {len(scored):,}")

print("\nby library (recovered variant effects):")
print(scored[is_new].groupby("library").size().sort_values(
    ascending=False).to_string())

print("\nper-file detail:")
print(per_file.to_string(index=False))

variant effects by mutation type

                      variant_effects  distinct_variants  median_barcodes
Mutation Type                                                            
missense                       430858             132472             25.0
synonymous wild type            21755               6687             24.0
deletion                        20777               6406             45.0
frame shift                     20703               7596             22.0
nonsense                        17235               6927             27.0
delins_2for1                    12474               4578             27.0
multi_change                     7165               4210              8.0
standard                          755                 12             76.0
wild type                          64                  1            355.5
unrecognised                        8                  4             12.5
deletion_multi                      3                  3              8.0

rec

## Step 9 — join the annotations and write

The recovered variants have no annotation rows of their own, but almost every
annotation is a property of a **position**, not of a
substitution — secondary structure, solvent accessibility, domain, interface,
HSP90/CDC37 contact, pLDDT. Those transfer to a recovered variant at the same
position in the same library. Variant-level annotations (the HSP90 buffering
metrics, ClinVar, AlphaMissense) do not, and are left null.

For a `delins` the position used is the first of the two residues, which is where
the fused codon sits.

In [9]:
# Stage 1 emits scores and identity only. Annotations are rebuilt from primary
# sources in stage 2 (Annotations.ipynb -> scripts/reannotate_scores.py).
out = scored.copy()

# ---------------------------------------------------------------------------
# Final reporting categories. `Mutation Type` keeps the fine-grained class the
# identity layer assigns; `variant_category` collapses it to the seven a reader
# needs, so every scored row falls in exactly one and none is unlabelled:
#
#   WT             no coding change
#   synonymous     codon changed, residue unchanged
#   missense       one residue substituted
#   3nt deletion   exactly one codon's worth of sequence gone. Codon-aligned it
#                  removes one residue (`A350-`); straddling a codon boundary the
#                  same 3-nt event fuses the flanks, taking two residues out and
#                  putting one new one in (`Y264_P265delinsS`). Both are 3-nt
#                  deletions and both are counted here -- the identities stay
#                  distinct, because the protein products differ.
#   nonsense       a stop introduced -- including a 3-nt deletion whose fused
#                  codon is a stop, which `parse_variant` names `X`n`*`
#   frameshift     reading frame broken
#   other          real, genotyped, but none of the above. Mostly multi-mutants
#                  (two or more independent substitutions, largely cloning/PCR
#                  artifacts), the rest deletion-containing --
#                  in-frame deletions of several residues, and deletions carrying an
#                  unrelated substitution elsewhere in the ORF. Deliberately NOT
#                  split further: only the single-codon deletion is a clean,
#                  interpretable class, so everything larger stays here.
#
# `standard` is kept separate because the spiked BRAF controls are not library
# variants. Rows whose genotype is unknown were dropped in Step 6, so there is
# no "unknown" bucket by construction.
CATEGORY = {
    "wild type": "WT",
    "synonymous wild type": "synonymous",
    "missense": "missense",
    "deletion": "3nt deletion",
    # A 3-nt deletion straddling a codon boundary: 2 residues out, 1 new in.
    # The same single-codon DNA event as `A350-`, merely out of frame with the
    # codon grid, so it belongs in the same category. It is NOT a frameshift --
    # the reading frame downstream is intact. A stop-gaining one never reaches
    # here; `parse_variant` classifies it as nonsense.
    "delins_2for1": "3nt deletion",
    "nonsense": "nonsense",
    "frame shift": "frameshift",
    "standard": "standard",
    # everything below is genotyped but does not fit the six simple classes
    "deletion_multi": "other",
    "multi_change": "other",
    "multi_change_fs": "other",
    "unrecognised": "other",
}
unmapped = sorted(set(out["Mutation Type"]) - set(CATEGORY))
assert not unmapped, f"Mutation Type with no category: {unmapped}"
out["variant_category"] = out["Mutation Type"].map(CATEGORY)
assert out["variant_category"].notna().all()

print("variant_category tally")
_t = (out.groupby("variant_category")
      .agg(variant_effects=("variant", "size"),
           distinct_variants=("variant", "nunique")))
_order = ["WT", "synonymous", "missense", "3nt deletion", "nonsense",
          "frameshift", "other", "standard"]
print(_t.reindex([c for c in _order if c in _t.index]).to_string())
print(f"\nevery row categorised: {out['variant_category'].notna().all()}")

# ---------------------------------------------------------------------------
# Reference accessions. `hgvs_p` (attached in Step 3) is deliberately left
# unprefixed -- the accession is carried in its own columns instead, so a reader
# can join on either identifier without parsing the HGVS string.
#
# `uniprot_id` is the isoform whose sequence our positions were validated
# against, which is NOT always the UniProt canonical: KRAS is P01116-2 (KRAS4B),
# because canonical P01116-1 (KRAS4A) matches only 167 of our 189 positions.
# Note this can differ from `uniprot_accession`, which is the base accession and
# is isoform-agnostic.
#
# `ensembl_protein` (ENSP) and `refseq_protein` (NP_/XP_) are both verified
# character-for-character identical to that isoform -- not merely
# cross-referenced. Two of UniProt's own Ensembl cross-references failed that test
# and were replaced (BRAF, SHP2), and KSR1 has no identical Ensembl translation at
# all among its nine, so `ensembl_protein` is null there while RefSeq still
# resolves it (XP_011523731.1). See scripts/verify_ensembl_proteins.py,
# scripts/verify_refseq_transcripts.py and config/protein_accessions.yaml.
#
# Only PROTEIN accessions are emitted, deliberately. This is a protein-level assay
# built on recoded wild-type ORFs, so the library's nucleotide sequence is not any
# reference transcript -- a `c.` description against NM_/ENST would be wrong for
# every variant, and pacybara's own `hgvsc`/`codonChanges` are relative to the
# recoded construct. The matching transcript is kept in the config as provenance
# only, since it is how each protein record was verified.
#
# `mane_select` records whether our proteoform IS the MANE Select one. It is for
# 14 of 17; BRAF, KRAS and KSR1 deliberately are not -- KRAS because the library
# is KRAS4B while MANE is KRAS4A, and KSR1 because MANE is the 928-aa form where
# ours is the 923-aa UniProt canonical.
ACC = yaml.safe_load((ROOT / "config" / "protein_accessions.yaml").read_text())
_prot = out["library"].str.replace(r"_(nterm|cterm)$", "", regex=True)
out["uniprot_id"] = _prot.map(lambda p: (ACC.get(p) or {}).get("uniprot"))
out["ensembl_protein"] = _prot.map(
    lambda p: (ACC.get(p) or {}).get("ensembl_protein"))
out["refseq_protein"] = _prot.map(
    lambda p: (ACC.get(p) or {}).get("refseq_protein"))
out["mane_select"] = _prot.map(lambda p: (ACC.get(p) or {}).get("mane_select"))

print("\nHGVS and accessions")
print(f"  hgvs_p         : {int(out['hgvs_p'].notna().sum()):,} of {len(out):,}")
print(f"  uniprot_id     : {int(out['uniprot_id'].notna().sum()):,}")
print(f"  ensembl_protein: {int(out['ensembl_protein'].notna().sum()):,}"
      f"  (null for KSR1: no identical Ensembl translation)")
print(f"  refseq_protein : {int(out['refseq_protein'].notna().sum()):,}")
print(f"  on MANE Select : {int(out['mane_select'].fillna(False).sum()):,} rows; "
      f"not MANE: "
      f"{sorted(out.loc[~out['mane_select'].fillna(False), 'library'].unique())}")
print("  rows with no HGVS, by Mutation Type:")
print(out.loc[out["hgvs_p"].isna(), "Mutation Type"].value_counts().to_string())

out["variant_class"] = np.where(
    out["Mutation Type"].isin(NEW_CLASSES), "recovered", "original")
out["count_data_delivery"] = "260404 (April)"

path = OUT / "scores_masterframe_recovered.tsv"
out.to_csv(path, sep="\t", index=False)
print(f"\nwrote {len(out):,} variant effects x {out.shape[1]} columns")
print(f"  -> {path.relative_to(ROOT)}")

per_file.to_csv(OUT / "recovery_per_file.tsv", sep="\t", index=False)
summary.to_csv(OUT / "recovery_by_mutation_type.tsv", sep="\t")
print(f"  -> {(OUT / 'recovery_per_file.tsv').relative_to(ROOT)}")
print(f"  -> {(OUT / 'recovery_by_mutation_type.tsv').relative_to(ROOT)}")

print("\nfinal tally:")
print(f"  original classes : {int((out['variant_class'] == 'original').sum()):,}")
print(f"  recovered        : {int((out['variant_class'] == 'recovered').sum()):,}")
print(f"  total            : {len(out):,}")

variant_category tally


                  variant_effects  distinct_variants
variant_category                                    
WT                             64                  1
synonymous                  21755               6687
missense                   430858             132472
3nt deletion                33251              10984
nonsense                    17235               6927
frameshift                  20703               7596
other                        7176               4217
standard                      755                 12

every row categorised: True



HGVS and accessions
  hgvs_p         : 531,041 of 531,797
  uniprot_id     : 531,797
  ensembl_protein: 497,772  (null for KSR1: no identical Ensembl translation)
  refseq_protein : 531,797
  on MANE Select : 400,640 rows; not MANE: ['braf_cterm', 'braf_nterm', 'kras', 'ksr1_cterm', 'ksr1_nterm']
  rows with no HGVS, by Mutation Type:
Mutation Type
standard        755
multi_change      1



wrote 531,797 variant effects x 38 columns
  -> output/scoring/scores_masterframe_recovered.tsv
  -> output/scoring/recovery_per_file.tsv
  -> output/scoring/recovery_by_mutation_type.tsv

final tally:
  original classes : 491,444
  recovered        : 40,353
  total            : 531,797


## Step 11 — control barcodes and the dominant-negative thresholds

The DN threshold is the level the pathway sits at with **no variant protein
expressed at all**, recomputed here from the control barcodes so the number that
decides what counts as dominant negative is derived rather than supplied.

Two controls exist, and both are used — but not everywhere:

* `empty_vector_std` — no cassette. Present in every cell, and treated along
  with the library, so it is valid under any condition.
* `NoVar_std` — the older control. **Spiked-in standards are never treated**, so
  a NoVar barcode reports the *untreated* pathway whatever the cell's treatment
  label says. It may therefore only be pooled into untreated cells, and treated
  cells (SerumStarve, CIAR, HSP90i) take empty vector alone.

Pooling matters because empty vector alone is thin in some cells, and the cost of
ignoring the treatment rule is large: an EGFR SerumStarve threshold built from
NoVar sits far above the empty-vector one, because the standard was never
serum-starved, and would call almost every variant dominant negative. The printed
table below gives the per-cell barcode counts and both thresholds.

**The threshold is a percentile of resampled *variants*, not of raw barcodes.** A
variant's score is the mean over its barcodes, so the right null is the
distribution of a `BOOT_K`-barcode mean under an empty construct. Each draw takes
`BOOT_K` barcodes with replacement and averages them; the threshold is the
`PCT`th percentile of `BOOT_N` such draws, repeated over `N_SEEDS` seeds with the
median taken, since a low percentile of a few hundred draws is otherwise
seed-dependent.

Taking the percentile of the raw barcode scores instead answers a different
question — "how low does a single barcode go" — which no variant score ever is,
and it drops every threshold substantially.


In [10]:
# ev_frames is set by the scoring loop (or by finalize_scores.py); fall back to
# the per-cell files so this cell also runs standalone.
try:
    ev_all = pd.concat(ev_frames, ignore_index=True)
except NameError:
    import glob
    ev_all = pd.concat([pd.read_csv(f, sep="\t")
                        for f in sorted(glob.glob(str(OUT / "ev" / "*.tsv")))],
                       ignore_index=True)

# the CORRECTED control scores, so the threshold sits on the same scale as the
# variant scores it will be compared against
REP = [f"corrected_score_{j}" for j in (1, 2, 3)]
ev_all["average score"] = ev_all[REP].mean(axis=1)
ev_all = ev_all.dropna(subset=["average score"])

ctrl_path = OUT / "control_barcode_scores.tsv"
ev_all.to_csv(ctrl_path, sep="\t", index=False)
print(f"wrote {ctrl_path.relative_to(ROOT)}")
print(f"  {len(ev_all):,} control barcodes with a score")
print(ev_all.groupby("variant").size().rename("barcodes").to_frame().to_string())

BOOT_K = 10                              # resample size: a variant's barcodes
#: The threshold is a percentile of the draw distribution, so it is an order
#: statistic: the 2.5th percentile of 200 draws is the 5th smallest value, of 500
#: draws the ~13th. Estimating it from 13 order statistics rather than 5 is
#: markedly less seed-dependent, and the only cost is arithmetic.
BOOT_N = 500
#: 2.5% in one tail. The synonymous test is two-sided, so 2.5% each side there is
#: the same evidentiary bar as 2.5% in this one-sided tail.
PCT = 2.5
N_SEEDS = 200                            # repeats, to size our own uncertainty
MIN_BARCODES = 10
#: NoVar_std is a spiked-in standard and standards are never treated, so a NoVar
#: barcode reports the untreated pathway whatever the cell's label. It may only
#: join the pool in an untreated cell. Everything else takes empty vector alone.
UNTREATED = {"No_treatment", "DMSO"}
#: Pool the empty vector of a split library's two halves into one threshold?
#:
#: False (current) gives each half its own threshold. True pools them, on the
#: grounds that the halves are two scans of the same protein whose empty vectors
#: measure the same thing.
#:
#: It matters for BRAF only, whose halves disagree about where the baseline sits.
#: The cterm-only threshold lands inside the synonymous peak, so it calls
#: synonymous variants as dominant negative -- and since the synonymous rate
#: estimates the neutral false-positive rate, that is a large FDR for that
#: library. Pooling removes it, at the cost of some BRAF calls.
POOL_HALVES = False


def boot_draws(x, seed=0, n_boot=BOOT_N):
    """`n_boot` draws of a `BOOT_K`-barcode mean -- the distribution a *variant*
    score is drawn from under this control."""
    rng = np.random.default_rng(seed)
    return rng.choice(x, size=(n_boot, BOOT_K), replace=True).mean(axis=1)


def dn_threshold(x, seed=0, n_boot=BOOT_N, pct=None):
    """`pct`-th percentile of that draw distribution."""
    return float(np.percentile(boot_draws(x, seed=seed, n_boot=n_boot),
                               PCT if pct is None else pct))


def draw_quantiles(x, seed=0):
    """Five-number summary of the bootstrap draw distribution itself.

    Emitted so the figures can *show* the null the threshold comes from without
    re-running the bootstrap. A second implementation downstream is how a panel
    silently drifts from the classification it illustrates -- which has already
    happened once here.
    """
    q = np.percentile(boot_draws(x, seed=seed), [2.5, 25, 50, 75, 97.5])
    return dict(zip(("boot_p2.5", "boot_p25", "boot_median", "boot_p75",
                     "boot_p97.5"), map(float, q)))


def threshold_with_spread(x, pct=None):
    """The procedure repeated over N_SEEDS seeds, median taken. The 2.5-97.5
    range over seeds is the cost of the 200-draw choice -- where it is wide, the
    DN count from that cell is soft."""
    spread = np.array([dn_threshold(x, seed=sd, pct=pct) for sd in range(N_SEEDS)])
    lo, hi = np.percentile(spread, [2.5, 97.5])
    return float(np.median(spread)), float(lo), float(hi)


# The control pool is built per (PROTEIN, assay, treatment), not per library:
# the nterm and cterm halves of a split library are two scans of the same
# protein, and their empty vectors measure the same thing. Pooling them matters
# because the halves can disagree about where that baseline sits: BRAF cterm's
# threshold alone lands inside the synonymous peak and calls synonymous variants
# as dominant negative. Pooled it does not, and the validated DN R509Y is still
# called comfortably.
#
# Note this does NOT rescue BRAF's weak dynamic range -- BRAF wild type barely
# activates over the empty vector, so the reference is intrinsically close to
# wild-type for this protein. It only stops one half's baseline being taken as
# the whole story.
ev_all["_protein"] = (ev_all.library.str.replace(r"_(nterm|cterm)$", "",
                                                 regex=True)
                      if POOL_HALVES else ev_all.library)

rows = []
for (prot, assay, treat), pp in ev_all.groupby(["_protein", "assay",
                                                "assay_treatment"]):
    ev = pp[pp.variant == "empty_vector_std"]
    nv = pp[pp.variant == "NoVar_std"]
    untreated = treat in UNTREATED
    # A library with only a handful of NoVar barcodes did not receive the
    # spike-in at all. Those few are index hopping or
    # carry-over from other libraries on the same run, not a control, so they
    # are treated as absent rather than pooled. Same floor as everywhere else.
    has_novar = len(nv) >= MIN_BARCODES
    use = pd.concat([ev, nv]) if (untreated and has_novar) else ev
    x_used = use["average score"].to_numpy()
    shared = None
    if len(x_used) >= MIN_BARCODES:
        thr, lo, hi = threshold_with_spread(x_used)
        shared = {"n_barcodes": len(x_used), "dn_threshold": thr,
                  "seed_lo": lo, "seed_hi": hi, "seed_width": hi - lo,
                  "median_barcode": float(np.median(x_used)),
                  "pooled_novar": bool(untreated and has_novar),
                  **draw_quantiles(x_used)}

    # one row per library, carrying the protein-level threshold, plus the
    # per-library single-control values kept as diagnostics
    for lib, gg in pp.groupby("library"):
        if shared is not None:
            rows.append({"library": lib, "assay": assay,
                         "assay_treatment": treat, "control": "used",
                         "pool_protein": prot,
                         "pooled_halves": pp.library.nunique() > 1,
                         "n_libraries_pooled": int(pp.library.nunique()),
                         **shared})
        for ctrl, g in (("empty_vector_std",
                         gg[gg.variant == "empty_vector_std"]),
                        ("NoVar_std", gg[gg.variant == "NoVar_std"])):
            x = g["average score"].to_numpy()
            if len(x) < MIN_BARCODES:
                continue
            t_, l_, h_ = threshold_with_spread(x)
            rows.append({"library": lib, "assay": assay,
                         "assay_treatment": treat, "control": ctrl,
                         "pool_protein": prot, "pooled_halves": False,
                         "n_libraries_pooled": 1,
                         "n_barcodes": len(x), "dn_threshold": t_,
                         "seed_lo": l_, "seed_hi": h_, "seed_width": h_ - l_,
                         "median_barcode": float(np.median(x)),
                         "pooled_novar": False, **draw_quantiles(x)})

thresholds = pd.DataFrame(rows).sort_values(
    ["control", "assay", "library", "assay_treatment"])
thr_path = OUT / "dn_thresholds_recomputed.tsv"
thresholds.to_csv(thr_path, sep="\t", index=False)
print(f"\nwrote {thr_path.relative_to(ROOT)}  ({len(thresholds)} cell x control)")

# --- what the rule did, and the evidence for it ---------------------------
_w = thresholds[thresholds.assay == "activity"].pivot_table(
    index=["library", "assay_treatment"], columns="control",
    values="dn_threshold")
if {"empty_vector_std", "NoVar_std"} <= set(_w.columns):
    _b = _w.dropna(subset=["empty_vector_std", "NoVar_std"]).copy()
    _b["ratio"] = _b.NoVar_std / _b.empty_vector_std
    _b["untreated"] = [t in UNTREATED for _l, t in _b.index]
    print("\nNoVar vs empty-vector threshold, cells carrying both:")
    print(_b[["empty_vector_std", "NoVar_std", "ratio", "untreated"]]
          .sort_values("ratio").to_string(float_format=lambda v: f"{v:9.3f}"))
    _u = _b[_b.untreated]
    print(f"\n  untreated cells: median ratio {_u.ratio.median():.2f}, "
          f"{int(((_u.ratio > 0.8) & (_u.ratio < 1.25)).sum())}/{len(_u)} agree "
          "within 20% -- these are pooled")
    _t = _b[~_b.untreated]
    if len(_t):
        print(f"  treated cells:   {len(_t)} NOT pooled (standards are never "
              f"treated); ratio " + ", ".join(f"{r:.1f}x" for r in _t.ratio))

act = thresholds[(thresholds.assay == "activity")
                 & (thresholds.control == "used")]
print(f"\nthresholds in use -- {PCT}th pct of {BOOT_N} draws of a {BOOT_K}-barcode "
      f"mean, median over {N_SEEDS} seeds:")
_sw = act.set_index(["library", "assay_treatment"])
print(f"  seed-to-seed spread: median "
      f"{(100 * _sw.seed_width / _sw.dn_threshold).median():.2f}% of the "
      f"threshold, from {N_SEEDS} repeats of the whole procedure.")
print(f"  ({BOOT_N} draws puts the {PCT}th percentile at the ~"
      f"{int(round(BOOT_N * PCT / 100))}th order statistic.)")
_ev = thresholds[(thresholds.assay == "activity")
                 & (thresholds.control == "empty_vector_std")].set_index(
                     ["library", "assay_treatment"])
_show = act.set_index(["library", "assay_treatment"]).copy()
_show["EV_only_n"] = _ev.n_barcodes
_show["EV_only_thr"] = _ev.dn_threshold
print(_show[["n_barcodes", "EV_only_n", "dn_threshold", "EV_only_thr",
             "seed_lo", "seed_hi", "pooled_novar", "pooled_halves"]]
      .to_string(float_format=lambda v: f"{v:8.4f}"))


wrote output/scoring/control_barcode_scores.tsv
  84,345 control barcodes with a score
                  barcodes
variant                   
NoVar_std             8074
empty_vector_std     76271



wrote output/scoring/dn_thresholds_recomputed.tsv  (127 cell x control)

NoVar vs empty-vector threshold, cells carrying both:
control                     empty_vector_std  NoVar_std     ratio  untreated
library    assay_treatment                                                  
ksr1_nterm No_treatment                3.922      3.398     0.866       True
araf_nterm No_treatment                0.546      0.494     0.906       True
ksr2_nterm No_treatment                1.140      1.055     0.926       True
erbb2      No_treatment                0.345      0.324     0.937       True
ret        No_treatment                0.778      0.732     0.940       True
ksr1_cterm No_treatment                5.485      5.157     0.940       True
grb2       No_treatment                1.258      1.195     0.950       True
shp2       No_treatment                0.798      0.764     0.958       True
araf_cterm No_treatment                0.618      0.595     0.962       True
sos2       No_treatment  